### Hyperparameter Tuning with **Optuna**

In [9]:
# ======================================================
# 1. IMPORT LIBRARIES
# ======================================================

import optuna
from sklearn.model_selection import cross_val_score

import pandas as pd
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [10]:
# ======================================================
# 2. LOAD DATASET
# ======================================================

df = sns.load_dataset("titanic")

df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [11]:
# ======================================================
# 3. SELECT USEFUL COLUMNS
# ======================================================

df = df[['pclass','sex','age','fare','survived']]

In [12]:
# ======================================================
# 4. HANDLE MISSING VALUES
# ======================================================

df.dropna(inplace=True)

In [13]:
# ======================================================
# 5. ENCODING
# ======================================================

le = LabelEncoder()

df['sex'] = le.fit_transform(df['sex'])

# male = 1
# female = 0

In [14]:
# ======================================================
# 6. FEATURES & TARGET
# ======================================================

X = df.drop('survived', axis=1)

y = df['survived']

In [15]:
# ======================================================
# 7. TRAIN TEST SPLIT
# ======================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [16]:
# ======================================================
# 8. FEATURE SCALING
# ======================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)

In [17]:
# ======================================================
# 9. MODEL BEFORE HYPERPARAMETER TUNING
# ======================================================

baseline_model = RandomForestClassifier(random_state=42)

baseline_model.fit(X_train,y_train)

y_pred_before = baseline_model.predict(X_test)

print("\n")
print("========== BEFORE TUNING ==========")

print("Accuracy:",accuracy_score(y_test,y_pred_before))
print("Precision:",precision_score(y_test,y_pred_before))
print("Recall:",recall_score(y_test,y_pred_before))
print("F1 Score:",f1_score(y_test,y_pred_before))

print("\nClassification Report\n")

print(classification_report(y_test,y_pred_before))



========== BEFORE TUNING ==========
Accuracy: 0.7832167832167832
Precision: 0.7272727272727273
Recall: 0.7142857142857143
F1 Score: 0.7207207207207207

Classification Report

              precision    recall  f1-score   support

           0       0.82      0.83      0.82        87
           1       0.73      0.71      0.72        56

    accuracy                           0.78       143
   macro avg       0.77      0.77      0.77       143
weighted avg       0.78      0.78      0.78       143



In [18]:
# ======================================================
# 10. Optuna
# ======================================================

def objective(trial):

    params = {
        'n_estimators': trial.suggest_int('n_estimators',100,700),
        'max_depth': trial.suggest_int('max_depth',5,25),
        'min_samples_split': trial.suggest_int('min_samples_split',2,15),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf',1,6),
        'max_features': trial.suggest_categorical('max_features',['sqrt','log2'])
    }

    model = RandomForestClassifier(**params,random_state=42)
    score = cross_val_score(model,X_train,y_train,cv=5,scoring='accuracy').mean()

    return score

study = optuna.create_study(direction='maximize')
study.optimize(objective,n_trials=50)

[I 2026-06-23 00:18:15,598] A new study created in memory with name: no-name-c8b3c8f3-6a84-4c65-8cbb-3252b941c064
[I 2026-06-23 00:18:20,491] Trial 0 finished with value: 0.8371777269260106 and parameters: {'n_estimators': 611, 'max_depth': 18, 'min_samples_split': 12, 'min_samples_leaf': 6, 'max_features': 'log2'}. Best is trial 0 with value: 0.8371777269260106.
[I 2026-06-23 00:18:22,161] Trial 1 finished with value: 0.8248970251716248 and parameters: {'n_estimators': 233, 'max_depth': 11, 'min_samples_split': 6, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 0 with value: 0.8371777269260106.
[I 2026-06-23 00:18:23,787] Trial 2 finished with value: 0.8284057971014492 and parameters: {'n_estimators': 250, 'max_depth': 14, 'min_samples_split': 9, 'min_samples_leaf': 4, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.8371777269260106.
[I 2026-06-23 00:18:27,882] Trial 3 finished with value: 0.8336536994660564 and parameters: {'n_estimators': 579, 'max_depth': 18, '

In [19]:
# ======================================================
# 11. BEST PARAMETERS
# ======================================================

print("\n")
print("========== BEST PARAMETERS ==========")

print(study.best_params)

print("\nBest CV Score:",study.best_value)



========== BEST PARAMETERS ==========
{'n_estimators': 697, 'max_depth': 24, 'min_samples_split': 2, 'min_samples_leaf': 6, 'max_features': 'log2'}

Best CV Score: 0.8389321128909228


In [25]:
# ======================================================
# 12. TRAIN MODEL WITH BEST PARAMETERS
# ======================================================

# best_model = RandomForestClassifier(**study.best_params,random_state=42)
best_model = RandomForestClassifier(n_estimators=697, max_depth=24, min_samples_split=2, min_samples_leaf=6, max_features='log2', random_state=42)

best_model.fit(X_train,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",697
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",24
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",6
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'log2'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y

In [26]:
# ======================================================
# 13. PREDICTIONS AFTER TUNING
# ======================================================

y_pred_after = best_model.predict(X_test)

In [27]:
# ======================================================
# 14. PERFORMANCE AFTER TUNING
# ======================================================

print("\n")
print("========== AFTER TUNING ==========")

accuracy_after = accuracy_score(y_test,y_pred_after)
precision_after = precision_score(y_test,y_pred_after)
recall_after = recall_score(y_test,y_pred_after)
f1_after = f1_score(y_test,y_pred_after)

print("Accuracy:",accuracy_after)
print("Precision:",precision_after)
print("Recall:",recall_after)
print("F1 Score:",f1_after)

print("\nClassification Report\n")
print(classification_report(y_test,y_pred_after))



========== AFTER TUNING ==========
Accuracy: 0.7692307692307693
Precision: 0.7169811320754716
Recall: 0.6785714285714286
F1 Score: 0.6972477064220184

Classification Report

              precision    recall  f1-score   support

           0       0.80      0.83      0.81        87
           1       0.72      0.68      0.70        56

    accuracy                           0.77       143
   macro avg       0.76      0.75      0.76       143
weighted avg       0.77      0.77      0.77       143



In [28]:
# ======================================================
# 15. CONFUSION MATRIX
# ======================================================

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test,y_pred_after))


Confusion Matrix

[[72 15]
 [18 38]]


In [29]:
# ======================================================
# 16. COMPARISON
# ======================================================

print("\n")
print("========== PERFORMANCE COMPARISON ==========")

print("Accuracy Improvement:",accuracy_after -accuracy_score(y_test,y_pred_before))
print("Precision Improvement:",precision_after -precision_score(y_test,y_pred_before))
print("Recall Improvement:",recall_after -recall_score(y_test,y_pred_before))
print("F1 Improvement:",f1_after -f1_score(y_test,y_pred_before))



========== PERFORMANCE COMPARISON ==========
Accuracy Improvement: -0.013986013986013957
Precision Improvement: -0.010291595197255643
Recall Improvement: -0.0357142857142857
F1 Improvement: -0.0234730142987023
